# Qwen3-VL ID Association Preliminary Analysis

This notebook implements four experiments on the MuPNIT-ReID test set.
- Chunk 1: Pairwise verifier
- Chunk 2: Folder-to-folder verifier
- Chunk 3: Profile builder + matcher (two profile types)
- Chunk 4: OCR-first association

All outputs are written under `id_assoc/results/` and reusable artifacts under `id_assoc/artifacts/`.

In [1]:
# Setup (run once)
import csv
import json
import os
import random
import re
from collections import defaultdict, Counter
from pathlib import Path
from typing import Dict, List, Tuple

from PIL import Image
from tqdm import tqdm
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from qwen_vl_utils import process_vision_info

DATASET_ROOT = Path("/4tb/dataset/NSVATrack_MuPNIT/MuPNIT-ReID/test")
PROJECT_ROOT = Path("/home/labad/minxing/code/Qwen3-VL/id_assoc")
RESULTS_DIR = PROJECT_ROOT / "results"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 2024
random.seed(RANDOM_SEED)

MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
DTYPE = "auto"
DEVICE_MAP = "auto"
FLASH_ATTN2 = False
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.2
TOP_P = 0.9
TOP_K = 40
REPETITION_PENALTY = 1.05
MIN_PIXELS = None
MAX_PIXELS = None

ENABLE_INFERENCE = True  # Set False to generate pair lists without model calls

CACHE_PATH = ARTIFACTS_DIR / "vlm_cache.json"
CACHE_EVERY = 50
SKIP_IF_EXISTS = True  # Skip recomputation if done markers exist
REUSE_PAIRS_IF_EXISTS = True

VLM_CACHE = {}
_CACHE_DIRTY = 0
if CACHE_PATH.exists():
    with CACHE_PATH.open("r", encoding="utf-8") as f:
        VLM_CACHE = json.load(f)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".heic", ".heif"}

def save_cache():
    if not VLM_CACHE:
        return
    with CACHE_PATH.open("w", encoding="utf-8") as f:
        json.dump(VLM_CACHE, f, ensure_ascii=True, indent=2)

def update_cache(cache_key: str, value: str):
    global _CACHE_DIRTY
    VLM_CACHE[cache_key] = value
    _CACHE_DIRTY += 1
    if _CACHE_DIRTY >= CACHE_EVERY:
        save_cache()
        _CACHE_DIRTY = 0

def make_cache_key(tag: str, items: List[str], prompt: str, extra: Dict = None) -> str:
    base = {"tag": tag, "items": items, "prompt": prompt}
    if extra:
        base["extra"] = extra
    return json.dumps(base, sort_keys=True)

def print_json_summary(path: Path, title: str):
    if not path.exists():
        print(f"{title}: summary not found at {path}")
        return
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"{title} summary:\n{json.dumps(data, indent=2)}")

def write_done_marker(path: Path):
    path.write_text('done')

def is_done(path: Path) -> bool:
    return path.exists() and path.read_text().strip() == 'done'

def parse_game_id(path: Path) -> str:
    stem = path.stem
    parts = stem.split("_")
    return parts[0] if parts else "unknown"

def index_dataset(root: Path, cache_path: Path) -> Dict:
    if cache_path.exists():
        with cache_path.open("r", encoding="utf-8") as f:
            return json.load(f)
    players = {}
    for player_dir in sorted(root.iterdir()):
        if not player_dir.is_dir():
            continue
        images = []
        games = defaultdict(list)
        for p in player_dir.rglob("*"):
            if p.suffix.lower() in IMG_EXTS:
                images.append(str(p))
                gid = parse_game_id(p)
                games[gid].append(str(p))
        if images:
            players[player_dir.name] = {
                "images": sorted(images),
                "games": {g: sorted(v) for g, v in games.items()},
            }
    data = {"players": players}
    with cache_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=True, indent=2)
    return data

DATA_INDEX = index_dataset(DATASET_ROOT, ARTIFACTS_DIR / "dataset_index.json")
PLAYERS = list(DATA_INDEX["players"].keys())
print(f"Indexed players: {len(PLAYERS)}")

_MODEL = None
_PROCESSOR = None

def get_model():
    global _MODEL, _PROCESSOR
    if _MODEL is not None and _PROCESSOR is not None:
        return _MODEL, _PROCESSOR
    dtype = "auto"
    if DTYPE.lower() in ("bfloat16", "bf16"):
        dtype = torch.bfloat16
    elif DTYPE.lower() in ("float16", "fp16", "half"):
        dtype = torch.float16
    elif DTYPE.lower() in ("float32", "fp32"):
        dtype = torch.float32
    attn_impl = "flash_attention_2" if FLASH_ATTN2 else "sdpa"
    _PROCESSOR = AutoProcessor.from_pretrained(MODEL_NAME)
    if hasattr(_PROCESSOR, "tokenizer"):
        _PROCESSOR.tokenizer.padding_side = "left"
    _MODEL = AutoModelForImageTextToText.from_pretrained(
        MODEL_NAME,
        dtype=dtype,
        attn_implementation=attn_impl,
        device_map=DEVICE_MAP,
    )
    _MODEL.eval()
    return _MODEL, _PROCESSOR

def load_image(path: str) -> Image.Image:
    return Image.open(path).convert("RGB")

def make_image_content(path: str) -> Dict:
    content = {"type": "image", "image": load_image(path)}
    if MIN_PIXELS is not None:
        content["min_pixels"] = MIN_PIXELS
    if MAX_PIXELS is not None:
        content["max_pixels"] = MAX_PIXELS
    return content

def run_vlm(messages: List[Dict], cache_key: str = None) -> str:
    if cache_key and cache_key in VLM_CACHE:
        return VLM_CACHE[cache_key]
    if not ENABLE_INFERENCE:
        return "[INFERENCE_DISABLED]"
    model, processor = get_model()
    texts = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos, video_kwargs = process_vision_info(
        messages, image_patch_size=16, return_video_kwargs=True, return_video_metadata=True
    )
    if videos is not None:
        videos, video_metadatas = zip(*videos)
        videos, video_metadatas = list(videos), list(video_metadatas)
    else:
        video_metadatas = None
    inputs = processor(
        text=texts,
        images=images,
        videos=videos,
        video_metadata=video_metadatas,
        padding=True,
        return_tensors="pt",
        do_resize=False,
        **video_kwargs,
    ).to(model.device)
    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        repetition_penalty=REPETITION_PENALTY,
    )
    with torch.no_grad():
        generated = model.generate(**inputs, **gen_kwargs)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    text = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    text = (text or "").strip()
    if cache_key:
        update_cache(cache_key, text)
    return text

def parse_yes_no_uncertain(text: str) -> Tuple[str, float]:
    t = text.lower()
    has_yes = re.search(r"\byes\b", t) is not None
    has_no = re.search(r"\bno\b", t) is not None
    has_uncertain = "uncertain" in t or "not sure" in t
    if has_uncertain or (has_yes and has_no):
        label = "uncertain"
    elif has_yes:
        label = "yes"
    elif has_no:
        label = "no"
    else:
        label = "unknown"
    conf = None
    m = re.search(r"confidence\s*[:=]?\s*([01](?:\.\d+)?)", t)
    if m:
        try:
            conf = float(m.group(1))
        except ValueError:
            conf = None
    return label, conf if conf is not None else -1.0

def write_jsonl(path: Path, rows: List[Dict]):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=True) + "\n")

def infer_fieldnames(rows: List[Dict]) -> List[str]:
    if not rows:
        return []
    fields = list(rows[0].keys())
    for row in rows[1:]:
        for key in row.keys():
            if key not in fields:
                fields.append(key)
    return fields

def write_csv(path: Path, rows: List[Dict], fieldnames: List[str] = None):
    fields = fieldnames if fieldnames else infer_fieldnames(rows)
    if not fields:
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

def summarize_binary_results(rows: List[Dict], label_key: str, positive_values: List[str]):
    total = len(rows)
    correct = 0
    unknown = 0
    tp = fp = tn = fn = 0
    by_type = defaultdict(Counter)
    for r in rows:
        expected_yes = r[label_key] in positive_values
        pred = r["parsed_label"]
        if pred in ("unknown", "uncertain"):
            unknown += 1
        else:
            is_yes = pred == "yes"
            correct += int(is_yes == expected_yes)
            if is_yes and expected_yes:
                tp += 1
            elif is_yes and not expected_yes:
                fp += 1
            elif (not is_yes) and expected_yes:
                fn += 1
            else:
                tn += 1
        by_type[r[label_key]][pred] += 1
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    specificity = tn / max(1, tn + fp)
    summary = {
        "total": total,
        "correct": correct,
        "unknown_or_uncertain": unknown,
        "accuracy_excluding_uncertain": (correct / max(1, total - unknown)),
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "confusion": {
            "tp": tp,
            "fp": fp,
            "tn": tn,
            "fn": fn,
        },
        "counts_by_type": {k: dict(v) for k, v in by_type.items()},
    }
    return summary

def release_model():
    global _MODEL, _PROCESSOR
    _MODEL = None
    _PROCESSOR = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def sample_two(paths: List[str]) -> Tuple[str, str]:
    a, b = random.sample(paths, 2)
    return a, b

def sample_same_player_same_game(n_pairs: int) -> List[Tuple[str, str, str, str]]:
    pairs = []
    attempts = 0
    while len(pairs) < n_pairs and attempts < n_pairs * 50:
        attempts += 1
        player = random.choice(PLAYERS)
        games = DATA_INDEX["players"][player]["games"]
        valid_games = [g for g, imgs in games.items() if len(imgs) >= 2]
        if not valid_games:
            continue
        game_id = random.choice(valid_games)
        img_a, img_b = sample_two(games[game_id])
        pairs.append((player, game_id, img_a, img_b))
    return pairs

def sample_same_player_diff_game(n_pairs: int) -> List[Tuple[str, str, str, str, str]]:
    pairs = []
    attempts = 0
    while len(pairs) < n_pairs and attempts < n_pairs * 50:
        attempts += 1
        player = random.choice(PLAYERS)
        games = DATA_INDEX["players"][player]["games"]
        if len(games) < 2:
            continue
        g1, g2 = random.sample(list(games.keys()), 2)
        img_a = random.choice(games[g1])
        img_b = random.choice(games[g2])
        pairs.append((player, g1, g2, img_a, img_b))
    return pairs

def sample_diff_player(n_pairs: int) -> List[Tuple[str, str, str]]:
    pairs = []
    attempts = 0
    while len(pairs) < n_pairs and attempts < n_pairs * 50:
        attempts += 1
        p1, p2 = random.sample(PLAYERS, 2)
        img_a = random.choice(DATA_INDEX["players"][p1]["images"])
        img_b = random.choice(DATA_INDEX["players"][p2]["images"])
        pairs.append((p1, p2, img_a, img_b))
    return pairs

PAIRWISE_PROMPT = (
    "Are these two crops the same player identity? Answer Yes, No, or Uncertain. "
    "Provide a confidence score (0-1) and list up to 3 visual evidences."
)

PROFILE_PROMPT_GAME = (
    "Summarize stable identity cues for this player in this game. "
    "Focus on body build, limb proportions, face shape, hair style, gait/posture, "
    "and any distinctive marks. Avoid jersey number/team colors unless they are the only cues."
)

PROFILE_PROMPT_CROSS = (
    "Summarize identity cues that persist across games. Avoid jersey number, "
    "team colors, and transient clothing. Focus on body build, proportions, "
    "face/head shape, hair pattern, gait, and distinctive marks."
)

PROFILE_MATCH_PROMPT = (
    "Given the identity profile below, decide if the query crop is the same player. "
    "Answer Yes/No/Uncertain with confidence 0-1 and evidence."
)

OCR_PROMPT = (
    "Extract any visible jersey numbers or readable text on the player. "
    "Return only the number(s) or 'none'."
)


/home/labad/miniconda3/envs/qwen3vl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Indexed players: 51


## Chunk 1: Pairwise verifier

Experiment: sample three groups of pairs (1000 each):
- Same player, same game ID (same outfit/appearance context)
- Same player, different game IDs (appearance changes across seasons/games)
- Different players (negative pairs)

We prompt Qwen3-VL with two crops to decide if they are the same identity.
Expected: high Yes on same-game pairs, lower Yes on different-player pairs, and the
same-player/different-game group should be the hardest.

In [2]:
# Pairwise verifier
PAIR_COUNT = 1000

pairs_path = RESULTS_DIR / "pairwise_pairs.jsonl"
results_path = RESULTS_DIR / "pairwise_results.csv"
summary_path = RESULTS_DIR / "pairwise_summary.json"
done_path = RESULTS_DIR / "pairwise.done"

if SKIP_IF_EXISTS and is_done(done_path):
    print(f"Skipping pairwise: found done marker {done_path}")
    print_json_summary(summary_path, 'Pairwise')
else:
    same_game_pairs = sample_same_player_same_game(PAIR_COUNT)
    diff_game_pairs = sample_same_player_diff_game(PAIR_COUNT)
    diff_player_pairs = sample_diff_player(PAIR_COUNT)

    pair_rows = []
    pair_rows.extend([
        {"pair_type": "same_player_same_game", "player": p, "game_id": gid, "img_a": a, "img_b": b}
        for p, gid, a, b in same_game_pairs
    ])
    pair_rows.extend([
        {"pair_type": "same_player_diff_game", "player": p, "game_a": g1, "game_b": g2, "img_a": a, "img_b": b}
        for p, g1, g2, a, b in diff_game_pairs
    ])
    pair_rows.extend([
        {"pair_type": "diff_player", "player_a": p1, "player_b": p2, "img_a": a, "img_b": b}
        for p1, p2, a, b in diff_player_pairs
    ])

    write_jsonl(pairs_path, pair_rows)
    print(f"Saved pairs to {pairs_path}")

    results = []
    for row in tqdm(pair_rows, desc="Pairwise"):
        key_items = sorted([row["img_a"], row["img_b"]])
        cache_key = make_cache_key("pairwise", key_items, PAIRWISE_PROMPT)
        messages = [
            {
                "role": "user",
                "content": [
                    make_image_content(row["img_a"]),
                    make_image_content(row["img_b"]),
                    {"type": "text", "text": PAIRWISE_PROMPT},
                ],
            }
        ]
        answer = run_vlm(messages, cache_key=cache_key)
        parsed, conf = parse_yes_no_uncertain(answer)
        results.append({
            **row,
            "answer": answer,
            "parsed_label": parsed,
            "confidence": conf,
        })

    write_csv(results_path, results)
    summary = summarize_binary_results(results, label_key="pair_type", positive_values=["same_player_same_game", "same_player_diff_game"])
    with summary_path.open("w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=True, indent=2)
    save_cache()
    print_json_summary(summary_path, 'Pairwise')
    write_done_marker(done_path)
release_model()


Skipping pairwise: found done marker /home/labad/minxing/code/Qwen3-VL/id_assoc/results/pairwise.done
Pairwise summary:
{
  "total": 3000,
  "correct": 2242,
  "unknown_or_uncertain": 290,
  "accuracy_excluding_uncertain": 0.8273062730627306,
  "counts_by_type": {
    "same_player_same_game": {
      "yes": 891,
      "uncertain": 34,
      "no": 75
    },
    "same_player_diff_game": {
      "no": 214,
      "yes": 671,
      "uncertain": 115
    },
    "diff_player": {
      "no": 680,
      "uncertain": 141,
      "yes": 179
    }
  }
}


## Chunk 2: Folder-to-folder verifier

Experiment: compare player folders directly by sampling 3 random crops from each folder
and asking Qwen3-VL if the two groups belong to the same identity.

We evaluate both same-folder (positive) and different-folder (negative) pairs.
Expected: higher match confidence for same-folder pairs, lower for different players.

In [3]:
# Folder-to-folder verifier
SAMPLES_PER_PLAYER = 3
INCLUDE_SELF = True
MAX_PLAYER_PAIRS = 5000  # cap for feasibility

pairs_path = RESULTS_DIR / "folder_pairs.jsonl"
results_path = RESULTS_DIR / "folder_results.csv"
summary_path = RESULTS_DIR / "folder_summary.json"
done_path = RESULTS_DIR / "folder.done"

if SKIP_IF_EXISTS and is_done(done_path):
    print(f"Skipping folder-to-folder: found done marker {done_path}")
    print_json_summary(summary_path, 'Folder-to-folder')
else:
    player_pairs = []
    players = PLAYERS[:]
    for i, p1 in enumerate(players):
        start = i if INCLUDE_SELF else i + 1
        for j in range(start, len(players)):
            p2 = players[j]
            player_pairs.append((p1, p2))

    if len(player_pairs) > MAX_PLAYER_PAIRS:
        player_pairs = random.sample(player_pairs, MAX_PLAYER_PAIRS)

    pair_rows = []
    for p1, p2 in player_pairs:
        imgs1 = DATA_INDEX["players"][p1]["images"]
        imgs2 = DATA_INDEX["players"][p2]["images"]
        if len(imgs1) < SAMPLES_PER_PLAYER or len(imgs2) < SAMPLES_PER_PLAYER:
            continue
        s1 = random.sample(imgs1, SAMPLES_PER_PLAYER)
        s2 = random.sample(imgs2, SAMPLES_PER_PLAYER)
        pair_rows.append({"player_a": p1, "player_b": p2, "imgs_a": s1, "imgs_b": s2})

    write_jsonl(pairs_path, pair_rows)
    print(f"Saved pairs to {pairs_path}")

    results = []
    for row in tqdm(pair_rows, desc="Folder2Folder"):
        key_items = sorted(row["imgs_a"] + row["imgs_b"])
        cache_key = make_cache_key("folder", key_items, PAIRWISE_PROMPT)
        content = []
        for p in row["imgs_a"]:
            content.append(make_image_content(p))
        for p in row["imgs_b"]:
            content.append(make_image_content(p))
        content.append({"type": "text", "text": PAIRWISE_PROMPT})
        messages = [{"role": "user", "content": content}]
        answer = run_vlm(messages, cache_key=cache_key)
        parsed, conf = parse_yes_no_uncertain(answer)
        results.append({
            **row,
            "answer": answer,
            "parsed_label": parsed,
            "confidence": conf,
            "pair_type": "same_player" if row["player_a"] == row["player_b"] else "diff_player",
        })

    write_csv(results_path, results)
    summary = summarize_binary_results(results, label_key="pair_type", positive_values=["same_player"])
    with summary_path.open("w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=True, indent=2)
    save_cache()
    print_json_summary(summary_path, 'Folder-to-folder')
    write_done_marker(done_path)
release_model()


Skipping folder-to-folder: found done marker /home/labad/minxing/code/Qwen3-VL/id_assoc/results/folder.done
Folder-to-folder summary:
{
  "total": 1326,
  "correct": 1009,
  "unknown_or_uncertain": 88,
  "accuracy_excluding_uncertain": 0.8150242326332795,
  "counts_by_type": {
    "same_player": {
      "no": 3,
      "yes": 47,
      "uncertain": 1
    },
    "diff_player": {
      "yes": 226,
      "no": 962,
      "uncertain": 87
    }
  }
}


## Chunk 3: Profile builder + matcher

Two profile types are built with Qwen3-VL summaries, then used to verify queries:
- Type 1 (per-game profile): 5 crops from one game ID for a player.
  Evaluate on (1) same game, (2) different games, (3) other players.
- Type 2 (cross-game profile): 1 crop per game ID to capture cross-season identity cues.
  Evaluate on (1) self mixed queries, (2) other players, (3) mixed players.

Expected: Type 1 strong on same-game, weaker across games; Type 2 should be more
balanced across games but may be less confident overall.

In [4]:
# Profile builder + matcher
PROFILE_K = 5
QUERY_K = 10
MAX_PLAYERS = 200  # cap for feasibility

profiles1_path = RESULTS_DIR / "profiles_type1.jsonl"
eval1_path = RESULTS_DIR / "profile_eval_type1.csv"
summary1_path = RESULTS_DIR / "profile_summary_type1.json"
done1_path = RESULTS_DIR / "profile_type1.done"
profiles2_path = RESULTS_DIR / "profiles_type2.jsonl"
eval2_path = RESULTS_DIR / "profile_eval_type2.csv"
summary2_path = RESULTS_DIR / "profile_summary_type2.json"
done2_path = RESULTS_DIR / "profile_type2.done"

def build_profile(image_paths: List[str], prompt: str, tag: str) -> str:
    cache_key = make_cache_key(tag, sorted(image_paths), prompt)
    content = [make_image_content(p) for p in image_paths]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]
    return run_vlm(messages, cache_key=cache_key)

def match_profile(profile_text: str, image_path: str, tag: str) -> str:
    cache_key = make_cache_key(tag, [image_path], PROFILE_MATCH_PROMPT, extra={"profile": profile_text})
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": PROFILE_MATCH_PROMPT},
            {"type": "text", "text": "Profile: " + profile_text},
            make_image_content(image_path),
        ],
    }]
    return run_vlm(messages, cache_key=cache_key)

players = PLAYERS[:MAX_PLAYERS]

# Type 1: per-player per-game profile
if SKIP_IF_EXISTS and is_done(done1_path):
    print(f"Skipping profiles type1: found done marker {done1_path}")
    print_json_summary(summary1_path, 'Profiles Type1')
else:
    profile_rows = []
    eval_rows = []
    for player in tqdm(players, desc="Profiles-Type1"):
        games = DATA_INDEX["players"][player]["games"]
        for game_id, imgs in games.items():
            if len(imgs) < 2:
                continue
            profile_imgs = random.sample(imgs, min(PROFILE_K, len(imgs)))
            remaining = [p for p in imgs if p not in profile_imgs]
            if not remaining:
                continue
            profile_text = build_profile(profile_imgs, PROFILE_PROMPT_GAME, tag="profile_game")
            profile_rows.append({
                "profile_type": "per_game",
                "player": player,
                "game_id": game_id,
                "profile_images": profile_imgs,
                "profile_text": profile_text,
            })
            # (1) same game queries
            for q in random.sample(remaining, min(QUERY_K, len(remaining))):
                ans = match_profile(profile_text, q, tag="profile_match_game_same")
                pred, conf = parse_yes_no_uncertain(ans)
                eval_rows.append({
                    "profile_type": "per_game",
                    "eval_type": "same_game",
                    "player": player,
                    "game_id": game_id,
                    "query_image": q,
                    "answer": ans,
                    "parsed_label": pred,
                    "confidence": conf,
                })
            # (2) different game queries
            other_games = [g for g in games.keys() if g != game_id]
            if other_games:
                other_imgs = []
                for g in other_games:
                    other_imgs.extend(games[g])
                for q in random.sample(other_imgs, min(QUERY_K, len(other_imgs))):
                    ans = match_profile(profile_text, q, tag="profile_match_game_diff")
                    pred, conf = parse_yes_no_uncertain(ans)
                    eval_rows.append({
                        "profile_type": "per_game",
                        "eval_type": "diff_game",
                        "player": player,
                        "game_id": game_id,
                        "query_image": q,
                        "answer": ans,
                        "parsed_label": pred,
                        "confidence": conf,
                    })
            # (3) other players
            other_players = [p for p in players if p != player]
            for other in random.sample(other_players, min(QUERY_K, len(other_players))):
                other_img = random.choice(DATA_INDEX["players"][other]["images"])
                ans = match_profile(profile_text, other_img, tag="profile_match_game_other")
                pred, conf = parse_yes_no_uncertain(ans)
                eval_rows.append({
                    "profile_type": "per_game",
                    "eval_type": "other_player",
                    "player": player,
                    "game_id": game_id,
                    "query_image": other_img,
                    "answer": ans,
                    "parsed_label": pred,
                    "confidence": conf,
                })

    write_jsonl(profiles1_path, profile_rows)
    write_csv(eval1_path, eval_rows)
    summary = summarize_binary_results(eval_rows, label_key="eval_type", positive_values=["same_game", "diff_game"])
    with summary1_path.open("w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=True, indent=2)
    save_cache()
    print_json_summary(summary1_path, 'Profiles Type1')
    write_done_marker(done1_path)

# Type 2: cross-game profile
if SKIP_IF_EXISTS and is_done(done2_path):
    print(f"Skipping profiles type2: found done marker {done2_path}")
    print_json_summary(summary2_path, 'Profiles Type2')
else:
    profile_rows = []
    eval_rows = []
    for player in tqdm(players, desc="Profiles-Type2"):
        games = DATA_INDEX["players"][player]["games"]
        if len(games) < 2:
            continue
        profile_imgs = []
        for g, imgs in games.items():
            profile_imgs.append(random.choice(imgs))
        profile_text = build_profile(profile_imgs, PROFILE_PROMPT_CROSS, tag="profile_cross")
        profile_rows.append({
            "profile_type": "cross_game",
            "player": player,
            "profile_images": profile_imgs,
            "profile_text": profile_text,
        })
        # (1) self mixed queries
        all_imgs = DATA_INDEX["players"][player]["images"]
        remaining = [p for p in all_imgs if p not in profile_imgs]
        for q in random.sample(remaining, min(QUERY_K, len(remaining))):
            ans = match_profile(profile_text, q, tag="profile_match_cross_self")
            pred, conf = parse_yes_no_uncertain(ans)
            eval_rows.append({
                "profile_type": "cross_game",
                "eval_type": "self_mixed",
                "player": player,
                "query_image": q,
                "answer": ans,
                "parsed_label": pred,
                "confidence": conf,
            })
        # (2) 10 other players
        other_players = [p for p in players if p != player]
        for other in random.sample(other_players, min(10, len(other_players))):
            other_img = random.choice(DATA_INDEX["players"][other]["images"])
            ans = match_profile(profile_text, other_img, tag="profile_match_cross_other")
            pred, conf = parse_yes_no_uncertain(ans)
            eval_rows.append({
                "profile_type": "cross_game",
                "eval_type": "other_player",
                "player": player,
                "query_image": other_img,
                "answer": ans,
                "parsed_label": pred,
                "confidence": conf,
            })
        # (3) 10 players including himself, up to 10 crops each
        sample_players = [player] + random.sample(other_players, min(9, len(other_players)))
        for p in sample_players:
            imgs = DATA_INDEX["players"][p]["images"]
            for q in random.sample(imgs, min(QUERY_K, len(imgs))):
                ans = match_profile(profile_text, q, tag="profile_match_cross_mixed")
                pred, conf = parse_yes_no_uncertain(ans)
                eval_rows.append({
                    "profile_type": "cross_game",
                    "eval_type": "mixed_players",
                    "player": player,
                    "query_image": q,
                    "answer": ans,
                    "parsed_label": pred,
                    "confidence": conf,
                    "query_player": p,
                })

    write_jsonl(profiles2_path, profile_rows)
    write_csv(eval2_path, eval_rows)
    summary = summarize_binary_results(eval_rows, label_key="eval_type", positive_values=["self_mixed"])
    with summary2_path.open("w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=True, indent=2)
    save_cache()
    print_json_summary(summary2_path, 'Profiles Type2')
    write_done_marker(done2_path)

release_model()


Profiles-Type1:   0%|          | 0/51 [00:00<?, ?it/s]

Profiles-Type1: 100%|██████████| 51/51 [4:20:00<00:00, 305.89s/it]  


Profiles Type1 summary:
{
  "total": 4137,
  "correct": 1360,
  "unknown_or_uncertain": 1928,
  "accuracy_excluding_uncertain": 0.6156631960162969,
  "precision": 0.714572192513369,
  "recall": 0.7169684775318578,
  "specificity": 0.4052924791086351,
  "confusion": {
    "tp": 1069,
    "fp": 427,
    "tn": 291,
    "fn": 422
  },
  "counts_by_type": {
    "same_game": {
      "yes": 630,
      "uncertain": 620,
      "no": 227
    },
    "diff_game": {
      "yes": 439,
      "no": 195,
      "uncertain": 546
    },
    "other_player": {
      "uncertain": 762,
      "yes": 427,
      "no": 291
    }
  }
}


Profiles-Type2: 100%|██████████| 51/51 [3:07:47<00:00, 220.93s/it]  

Profiles Type2 summary:
{
  "total": 2520,
  "correct": 261,
  "unknown_or_uncertain": 1818,
  "accuracy_excluding_uncertain": 0.3717948717948718,
  "precision": 0.08974358974358974,
  "recall": 0.7368421052631579,
  "specificity": 0.3395348837209302,
  "confusion": {
    "tp": 42,
    "fp": 426,
    "tn": 219,
    "fn": 15
  },
  "counts_by_type": {
    "self_mixed": {
      "no": 15,
      "yes": 42,
      "uncertain": 153
    },
    "other_player": {
      "uncertain": 147,
      "no": 18,
      "yes": 45
    },
    "mixed_players": {
      "uncertain": 1518,
      "yes": 381,
      "no": 201
    }
  }
}


## Chunk 4: OCR-first association

Experiment: extract jersey numbers / readable text and use exact matches as a
lightweight association signal. We compare OCR matches across:
- Same player, same game (should often match if jersey visible)
- Same player, different games (numbers can change)
- Different players (should rarely match)

Expected: OCR helps most within the same game; performance drops across games.

In [ ]:
# OCR-first association
OCR_PAIR_COUNT = 1000

ocr_path = RESULTS_DIR / "ocr_results.csv"
summary_path = RESULTS_DIR / "ocr_summary.json"
done_path = RESULTS_DIR / "ocr.done"

if SKIP_IF_EXISTS and is_done(done_path):
    print(f"Skipping OCR: found done marker {done_path}")
    print_json_summary(summary_path, 'OCR')
else:
    def extract_ocr(path: str) -> str:
        cache_key = make_cache_key("ocr", [path], OCR_PROMPT)
        messages = [{
            "role": "user",
            "content": [make_image_content(path), {"type": "text", "text": OCR_PROMPT}],
        }]
        return run_vlm(messages, cache_key=cache_key)

    same_game_pairs = sample_same_player_same_game(OCR_PAIR_COUNT)
    diff_game_pairs = sample_same_player_diff_game(OCR_PAIR_COUNT)
    diff_player_pairs = sample_diff_player(OCR_PAIR_COUNT)

    rows = []
    for p, gid, a, b in tqdm(same_game_pairs, desc="OCR same_game"):
        ocr_a = extract_ocr(a)
        ocr_b = extract_ocr(b)
        match = (ocr_a.strip().lower() == ocr_b.strip().lower()) and ocr_a.strip().lower() != "none"
        rows.append({
            "pair_type": "same_player_same_game",
            "player": p,
            "game_id": gid,
            "img_a": a,
            "img_b": b,
            "ocr_a": ocr_a,
            "ocr_b": ocr_b,
            "ocr_match": match,
        })

    for p, g1, g2, a, b in tqdm(diff_game_pairs, desc="OCR diff_game"):
        ocr_a = extract_ocr(a)
        ocr_b = extract_ocr(b)
        match = (ocr_a.strip().lower() == ocr_b.strip().lower()) and ocr_a.strip().lower() != "none"
        rows.append({
            "pair_type": "same_player_diff_game",
            "player": p,
            "game_a": g1,
            "game_b": g2,
            "img_a": a,
            "img_b": b,
            "ocr_a": ocr_a,
            "ocr_b": ocr_b,
            "ocr_match": match,
        })

    for p1, p2, a, b in tqdm(diff_player_pairs, desc="OCR diff_player"):
        ocr_a = extract_ocr(a)
        ocr_b = extract_ocr(b)
        match = (ocr_a.strip().lower() == ocr_b.strip().lower()) and ocr_a.strip().lower() != "none"
        rows.append({
            "pair_type": "diff_player",
            "player_a": p1,
            "player_b": p2,
            "img_a": a,
            "img_b": b,
            "ocr_a": ocr_a,
            "ocr_b": ocr_b,
            "ocr_match": match,
        })

    write_csv(ocr_path, rows)
    summary = defaultdict(Counter)
    for r in rows:
        summary[r["pair_type"]]["total"] += 1
        summary[r["pair_type"]]["ocr_match"] += int(r["ocr_match"])
    summary = {k: dict(v) for k, v in summary.items()}
    with summary_path.open("w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=True, indent=2)
    save_cache()
    print_json_summary(summary_path, 'OCR')
    write_done_marker(done_path)
release_model()


OCR diff_player: 100%|██████████| 1000/1000 [17:04<00:00,  1.02s/it]

OCR summary:
{
  "same_player_same_game": {
    "total": 1000,
    "ocr_match": 191
  },
  "same_player_diff_game": {
    "total": 1000,
    "ocr_match": 170
  },
  "diff_player": {
    "total": 1000,
    "ocr_match": 22
  }
}


: 